# Phi-3-mini Financial Fine-tuning (QLoRA via Unsloth)

**Runtime:** Google Colab T4 (free tier)  
**Expected duration:** 30-45 min for 500 samples, 3 epochs  
**Output:** LoRA adapter pushed to HuggingFace Hub private repo

## Pipeline position
```
ragdb → export CronJob → JSONL on PVC
  → [Cell 2] Tailscale connects Colab to minicloud tailnet
  → [Cell 3] kubectl cp downloads JSONL from PVC to Colab
  → [Cell 5] Unsloth QLoRA trains Phi-3-mini
  → [Cell 5] MLflow logs run to mlflow.10.0.0.200.nip.io (via Tailscale)
  → [Cell 6] adapter pushed to HF Hub (andrelair-platform/phi3-financial-ft)
  → vLLM on-cluster loads adapter → LiteLLM routes phi3-financial-ft
```

## What you need
- **Tailscale auth key** — generate an ephemeral key at https://login.tailscale.com/admin/settings/keys  
  (Reusable: No, Ephemeral: Yes, Tag: none required)
- **HuggingFace write token** — https://huggingface.co/settings/tokens  
  Create a private repo `andrelair-platform/phi3-financial-ft` first
- The export CronJob must have run at least once (or trigger it manually)

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install unsloth mlflow huggingface_hub --quiet
!pip install --upgrade --no-cache-dir unsloth --quiet

In [ ]:
# ── 2. Connect Colab to the minicloud Tailscale tailnet ──────────────────────
# This gives Colab direct access to cluster-internal services:
#   - mlflow.10.0.0.200.nip.io  (MLflow tracking)
#   - harbor.10.0.0.200.nip.io  (container registry)
#   - 10.0.0.x nodes            (kubectl via controller)
#
# Generate a one-time ephemeral key at:
#   https://login.tailscale.com/admin/settings/keys

TS_AUTH_KEY = "tskey-auth-xxxx"   # <-- paste your key here

!curl -fsSL https://tailscale.com/install.sh | sh -s - --quiet
!tailscale up --authkey={TS_AUTH_KEY} --hostname=colab-phi3-training --accept-routes
!tailscale status

In [ ]:
# ── 3. Download training data from cluster PVC ───────────────────────────────
# The finetuning-data-export CronJob writes JSONL to /export/ on the
# finetuning-datasets PVC. This cell fetches the latest file via kubectl
# through the Tailscale-connected controller.
#
# If the CronJob hasn't run yet, trigger it manually first:
#   kubectl create job ft-export-manual --from=cronjob/finetuning-data-export -n ai

import subprocess, glob, os

CONTROLLER_IP = "100.88.123.8"   # Tailscale IP of controller

# Find the latest export pod (completed state)
result = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no",
     f"ktayl@{CONTROLLER_IP}",
     "kubectl get pod -n ai -l app=finetuning-data-export "
     "--field-selector=status.phase=Succeeded "
     "-o jsonpath='{.items[-1].metadata.name}'"],
    capture_output=True, text=True
)
pod_name = result.stdout.strip().strip("'")
print(f"Export pod: {pod_name}")

# List available JSONL files on the PVC
subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no",
     f"ktayl@{CONTROLLER_IP}",
     f"kubectl exec -n ai {pod_name} -- ls /export/"],
    check=True
)

# Copy the latest file to this Colab session
TRAINING_FILE = "training-data.jsonl"
subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no",
     f"ktayl@{CONTROLLER_IP}",
     f"kubectl exec -n ai {pod_name} -- cat /export/$(kubectl exec -n ai {pod_name} -- ls /export/ | tail -1)"],
    stdout=open(TRAINING_FILE, "w"), check=True
)

lines = open(TRAINING_FILE).readlines()
print(f"Downloaded {len(lines)} training examples to {TRAINING_FILE}")

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
HF_TOKEN   = "hf_xxxx"                              # HuggingFace write token
HF_REPO    = "andrelair-platform/phi3-financial-ft" # must be created first (private)
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# MLflow reaches the cluster via Tailscale (Cell 2 must have run)
MLFLOW_URI = "https://mlflow.10.0.0.200.nip.io"

# QLoRA hyperparams
LORA_RANK     = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
MAX_SEQ_LEN   = 1024
BATCH_SIZE    = 4
GRAD_ACCUM    = 4       # effective batch = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
WARMUP_STEPS  = 10

In [ ]:
# ── 5. Load Phi-3-mini with Unsloth 4-bit QLoRA ──────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ── 6. Format training data ───────────────────────────────────────────────────
import json
from datasets import Dataset

ALPACA_TEMPLATE = """Below is an instruction from a financial professional, with optional context.
Write a clear, accurate response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS = tokenizer.eos_token

records = []
with open(TRAINING_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

def format_record(row):
    return {"text": ALPACA_TEMPLATE.format(
        instruction=row["instruction"],
        input=row.get("input", ""),
        output=row["output"],
    ) + EOS}

dataset = Dataset.from_list(records).map(format_record)
print(f"Training examples: {len(dataset)}")
print("\nSample (first 300 chars):")
print(dataset[0]["text"][:300])

In [ ]:
# ── 7. Train + log to on-cluster MLflow (via Tailscale) ──────────────────────
import mlflow
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("phi3-financial-finetuning")

training_args = TrainingArguments(
    output_dir="/tmp/phi3-ft-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

with mlflow.start_run(run_name=f"phi3-financial-ft-r{LORA_RANK}") as run:
    mlflow.log_params({
        "base_model": MODEL_NAME,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRAD_ACCUM,
        "train_samples": len(dataset),
        "hf_repo": HF_REPO,
    })

    trainer_stats = trainer.train()

    mlflow.log_metrics({
        "train_loss": trainer_stats.training_loss,
        "train_runtime_min": trainer_stats.metrics["train_runtime"] / 60,
        "samples_per_second": trainer_stats.metrics["train_samples_per_second"],
    })

    RUN_ID = run.info.run_id
    print(f"MLflow run ID : {RUN_ID}")
    print(f"Train loss    : {trainer_stats.training_loss:.4f}")
    print(f"View run      : {MLFLOW_URI}/#/experiments/1/runs/{RUN_ID}")

In [ ]:
# ── 8. Quick inference check before push ─────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_TEMPLATE.format(
    instruction="What is the CET1 ratio requirement under Basel III?",
    input="",
    output="",
)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)[len(test_prompt):])

In [ ]:
# ── 9. Push adapter to HuggingFace Hub ───────────────────────────────────────
from huggingface_hub import login

login(token=HF_TOKEN)

commit_message = f"QLoRA adapter — mlflow_run={RUN_ID} loss={trainer_stats.training_loss:.4f}"

model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method="lora",
    token=HF_TOKEN,
    commit_message=commit_message,
    private=True,
)

# Update MLflow run with HF link
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("hf_adapter_commit", commit_message)
    mlflow.set_tag("adapter_ready", "true")
    mlflow.set_tag("hf_repo", f"https://huggingface.co/{HF_REPO}")

print(f"Adapter pushed → https://huggingface.co/{HF_REPO}")
print()
print("Next steps (in minicloud-gitops):")
print(f"  Edit 38-vllm.yaml — add to vLLM args:")
print(f"    - --lora-modules")
print(f"    - phi3-financial-ft={HF_REPO}")
print(f"  Open PR → ArgoCD syncs → vLLM hot-reloads the adapter")